# Concatenation, Merge, and GroupBy — Sales, IPL, and Iris (Task 20)

Task 20 is a consolidation exercise that mixes all the major Pandas skills: **concatenation** with hierarchical keys, **merging** related tables, **groupby** aggregation, custom per-row feature engineering, and plotting. Its three datasets are deliberately different in character — Chipotle-style restaurant sales split across quarterly files, a ball-by-ball IPL archive, and the classic Iris flowers split into sepal/petal/species fragments — and each stresses a distinct part of the toolkit.

The sales section (Q-1 to Q-5) opens with `pd.concat([q1, q2, q3], keys=['Q1','Q2','Q3'])`, which stacks the three quarterly DataFrames and stamps each block with a `Q1`/`Q2`/`Q3` outer index level. You then merge in the `items.csv` lookup to attach human-readable item names, and answer four business questions: the most-sold item per quarter, the highest-revenue item per quarter, and the average order price per quarter. The `item_price` column arrives as text with a `$` sign and must be converted to a number first. Each answer follows the same shape — group by the relevant keys, aggregate, sort, and take the top row per group with `drop_duplicates`.

The IPL section (Q-6 to Q-9) is the most demanding. Q-6 computes the **Purple Cap** winner each season: the bowler with the most wickets, tie-broken by the best economy, where economy is runs conceded per over. That requires engineering three derived columns — `IsBowlerWicket` (dismissal kinds credited to the bowler), `BowlerRun` (excluding byes and leg-byes), and `IsLegalBall` (excluding wides and no-balls) — before grouping by `(Season, bowler)`. Q-7 finds the best death-overs bowler (overs 16–20). Q-8 builds a per-season batting-record table — Innings, TotalRuns, Avg, HighestScore, StrikeRate — for any batter passed to a function, carefully handling how wides, no-balls, and run-outs affect balls faced and dismissals. Q-9 produces a match-by-match table of Player-of-the-Match batting and bowling figures in the compact forms `runs/balls` and `wickets/runs`.

The Iris section (Q-9 to Q-11) provisions the famous dataset from five fragments — sepal measurements, petal measurements, and three per-species tables — and practices multi-table **merging** and **concatenation**, plotting an average-length bar chart, assembling a shuffled complete dataset with `sample()`, and using `groupby` to summarise sepal and petal width extremes per species.

## 0. Setup

Import `pandas` and `numpy` to begin.

In [ ]:
import pandas as pd
import numpy as np

# 1. Quarterly Sales Data (Q-1 to Q-5)

Three quarterly order files plus an item-name lookup. The tasks stack, enrich, and summarise them.

DATA DESCRIPTION
```
file name -> Columns
quater-i.csv -> ['order_id', 'quantity', 'item_id', 'choice_description_id' 'item_price']
items.csv -> ['item_id', 'item_name']
```
Dataset Link - https://drive.google.com/drive/folders/1Z0kaFybvgFeczeUj4dldUnhTdloLqLsL?usp=share_link

## 1.0 Loading the data

`pd.read_csv` loads `quarter-1.csv`, `quarter-2.csv`, `quarter-3.csv`, and `items.csv` into separate objects. The quarterly files share a schema — `order_id`, `quantity`, `item_id`, `choice_description_id`, `item_price` — while `items.csv` maps `item_id` to a readable `item_name`. Notice that the quarterly files carry no explicit quarter marker; that information will be injected via the concatenation keys in the next step. Also note the `item_price` dtype: it arrives as text (`'$3.39'`), which is why every price calculation later needs a conversion step first. Establishing this up front — what each file holds, how they relate, and what needs cleaning — is exactly the reconnaissance that makes the joins and groups that follow go smoothly.

In [ ]:
# import like this
items_path = "items.csv"
q1_path = "quarter-1.csv"
q2_path = "quarter-2.csv"
q3_path = "quarter-3.csv"


q1= pd.read_csv(q1_path)
q2 = pd.read_csv(q2_path)
q3 = pd.read_csv(q3_path)

items = pd.read_csv(items_path)

# Tasks Q-1 & Q-2 — Concatenate with Quarter Keys

**Goal:** Append the three quarterly files into one DataFrame, with a Q-1/Q-2/Q-3 index level identifying the source quarter.

### `Q:1-5`
1. You are given three quater files, your job is to append these three files and make a single dataframe.
2. Have a index as Q-1 Q-2 Q-3 for respective quater files in the dataframe
3. Your are given a file items.csv which has item_id and item_name. Find out most sold items in each quarter.
4. Find out items which has made most revenue in each quarter.
5. Find out avg order price of each quarter.

***Note: item_price is given as str with $ sign, in earlier task you have converted this to rupees, here too first convert item_price field in rupees.***

## 1.1 pd.concat with keys

`pd.concat([q1, q2, q3], keys=['Q1','Q2','Q3'])` stacks the three frames vertically and, because `keys` is supplied, creates a **MultiIndex**: the outer level labels each block with its quarter and the inner level preserves each file's original row index. The printed frame shows exactly that — rows headered `Q1`, then `Q2`, then `Q3`, each with its own 0,1,2... inner index.

This is the canonical way to combine 'same-schema' files while remembering which file each row came from. Without `keys`, `pd.concat` would still stack the rows but you would lose the provenance; with `keys`, 'which quarter?' is answerable at a glance and usable as a grouping key. The `concat` default is `axis=0` (row-wise); `axis=1` would instead glue frames side by side by index. Other useful arguments include `ignore_index=True` (throw away the inner indices and renumber) and `join='inner'` (keep only shared columns). For this task you *want* the two-level index, so the default behaviour with `keys` is perfect.

In [ ]:
# code here
#1+2
df1 = pd.concat([q1, q2, q3], keys=["Q1", "Q2", "Q3"])
df1

        order_id quantity item_id choice_description_id item_price
Q1 0           1        1       1                     1     $3.39 
   1           1        1       2                     2     $3.39 
   2           2        2       4                     3    $16.98 
   3           4        1       7                     6     $9.25 
   4           6        1       9                     8     $8.75 
...          ...      ...     ...                   ...        ...
Q2 2342     1829        1      23                    92    $11.25 
   2343     1830        1      23                  1043    $11.25 
   2344     1832        1      10                   116     $8.75 
   2345     1832        1       8                     0     $4.45 
   2346     1834        1      20                   515    $11.25 

[4622 rows x 5 columns]

## 1.2 Most-sold items per quarter (Q-3)

The two-level index is inconvenient for merging, so `df1.reset_index()` moves the quarter key back into plain columns — `level_0` becomes the quarter. The call `new_df.merge(items, on='item_id')` then attaches `item_name` to every order line by matching `item_id`, and `rename(columns={'level_0':'quarter'})` gives the column a meaningful name.

The aggregation is:
```python
new_df.groupby(['quarter','item_name'], as_index=False)['quantity'].sum() \
    .sort_values('quantity', ascending=False).drop_duplicates(subset='quarter', keep='first')
```
Grouping by *both* quarter and item and summing `quantity` gives total units sold per item per quarter. Sorting descending, then `drop_duplicates(subset='quarter', keep='first')`, keeps only the top row for each quarter — a compact 'top-1 per group' idiom. The result: **Chicken Bowl** is the most-sold item in both Q2 (394) and Q1 (367). Alternatives to the sort-dedupe trick include `groupby(...).idxmax()` or `nlargest(1, ...)` per group, but note that `drop_duplicates` after a sort is often the clearest to read.

In [ ]:
#3
new_df = df1.reset_index().merge(items, on="item_id")
new_df.rename(columns={"level_0": "quarter"}, inplace=True)
new_df.groupby(["quarter", "item_name"], as_index = False)["quantity"].sum().sort_values("quantity", ascending = False).drop_duplicates(subset="quarter", keep="first")

   quarter     item_name  quantity
65      Q2  Chicken Bowl       394
17      Q1  Chicken Bowl       367

## 1.3 Highest-revenue items per quarter (Q-4)

Revenue needs a numeric price. The conversion `new_df['item_price'] = new_df.item_price.apply(lambda x: float(x[1:]))` strips the leading `$` (slice from index 1 onward) and parses the rest as a float. A vectorised alternative is `new_df['item_price'].str.replace('$','', regex=False).astype(float)`, but the apply version is intuitive for a lesson. Then `total_item_price = item_price * quantity` gives the line revenue.

The aggregation mirrors Q-3 but on revenue: group by `['quarter','item_name']`, sum `total_item_price`, sort descending, and `drop_duplicates('quarter', keep='first')` to keep the revenue leader per quarter. The output again shows **Chicken Bowl** on top — 4192.25 in Q2 and 3852.38 in Q1 — which makes sense because it is both the volume leader and a mid-to-high-priced item. The distinction between Q-3 and Q-4 is instructive: the most-sold item is not always the top earner, and computing both exposes that gap (here they coincide; elsewhere a cheap high-volume item can beat an expensive low-volume one).

In [ ]:
#4
new_df["item_price"] = new_df.item_price.apply(lambda x: float(x[1:]))
new_df["total_item_price"] = new_df["item_price"] * new_df["quantity"]
new_df.groupby(["quarter", "item_name"], as_index = False)["total_item_price"].sum().sort_values("total_item_price", ascending=False).drop_duplicates("quarter", keep="first")

   quarter     item_name  total_item_price
65      Q2  Chicken Bowl           4192.25
17      Q1  Chicken Bowl           3852.38

## 1.4 Average order price per quarter (Q-5)

An 'order' spans multiple line items sharing an `order_id`, so the average order price is a **two-level aggregation**: first sum the line revenues within each order, then average those order totals per quarter.
```python
new_df.groupby(['quarter','order_id'], as_index=False)['total_item_price'].sum() \
    .groupby('quarter', as_index=False)['total_item_price'].mean()
```
The first `groupby` collapses each order to one total; the second averages those totals per quarter, yielding Q1 ≈ 13.81 and Q2 ≈ 13.28. This 'aggregate, then aggregate again' pattern is essential whenever a metric is defined at a finer grain than the one you want to report — and it is easy to get wrong by averaging line items directly, which would weight large orders too heavily. `as_index=False` keeps the grouping keys as columns so the chained second groupby can reference them by name. The near-identical averages suggest stable pricing across the two quarters.

In [ ]:
#5
new_df.groupby(["quarter", "order_id"], as_index = False)["total_item_price"].sum().groupby("quarter", as_index = False)["total_item_price"].mean()

  quarter  total_item_price
0      Q1         13.809488
1      Q2         13.279828

# 2. IPL — Purple Cap, Death Overs, and Records (Q-6 to Q-9)

Two datasets are combined: `IPL_Ball_by_Ball_2008_2022.csv` (one row per delivery) and `IPL_Matches_2008_2022.csv` (one row per match, carrying the season). The season must be joined onto the ball data before any season-wise grouping.

### `Q-6` From the IPL dataset you have to find the Purple cap holder each season.

*Note: Bowler with most no wickets in a season gets purple cap. If more than one bowler have same no of wickets in the season, one with least ecomnomy among them is purple cap holder.*

Bowler's Economy = runs-conceded per six balls

## 2.0 The two IPL files

`balls` records every delivery of every match with columns such as `ID` (match), `innings`, `overs`, `batter`, `bowler`, `batsman_run`, `total_run`, `extra_type`, `kind`, `player_out`, and `isWicketDelivery`. `matches` holds match-level data including `ID`, `Season`, `Team1`, `Team2`, `WinningTeam`, and `Player_of_Match`. The ball file has no season column, so the very first move is `balls.merge(matches[['ID','Season']], on='ID')` — a many-to-one join that stamps each delivery with its season. Selecting only the needed `matches` columns before merging keeps the result lean. This is a standard star-schema pattern: a fact table (balls) enriched with dimensions (matches). Everything that follows depends on this join being correct, so verify the row count is unchanged (each ball finds exactly one match) before proceeding.

In [ ]:
balls = pd.read_csv("IPL_Ball_by_Ball_2008_2022.csv")
matches = pd.read_csv("IPL_Matches_2008_2022.csv")

## 2.1 Purple Cap per season (Q-6)

A Purple Cap goes to the season's leading wicket-taker, tie-broken by the best economy (runs conceded per over). Three derived columns encode the cricket rules:
```python
seasondf['IsBowlerWicket'] = seasondf.kind.apply(
    lambda x: 1 if x in ['caught','caught and bowled','bowled','stumped','lbw','hit wicket'] else 0)
seasondf['BowlerRun'] = seasondf.extra_type.apply(
    lambda x: 0 if x in ['legbyes','byes'] else 1) * seasondf['total_run']
seasondf['IsLegalBall'] = seasondf.extra_type.apply(
    lambda x: 0 if x in ['wides','noballs'] else 1)
```
`IsBowlerWicket` counts only dismissals credited to the bowler (excluding run-outs and the like). `BowlerRun` excludes byes and leg-byes, which are not the bowler's fault. `IsLegalBall` excludes wides and no-balls, which do not count toward an over. Grouping by `(Season, bowler)` and summing these gives each bowler's wickets, runs conceded, and legal balls per season; `Economy = BowlerRun / IsLegalBall * 6` converts to runs per over.

Finally `sort_values(['IsBowlerWicket','Economy'], ascending=[False, True]).drop_duplicates('Season', keep='first')` implements the tie-break exactly — most wickets first, best (lowest) economy among ties — and keeps one winner per season, yielding Sohail Tanvir (2007/08), RP Singh (2009), and PP Ojha (2009/10) among others, which match the historical Purple Cap list linked in the notebook.

In [ ]:
# code here
seasondf = balls.merge(matches[["ID", "Season"]], on="ID")
seasondf["IsBowlerWicket"] = seasondf.kind.apply(lambda x: 1 if x in ["caught", 'caught and bowled', 'bowled', 'stumped','lbw', 'hit wicket'] else 0)
seasondf["BowlerRun"] = seasondf.extra_type.apply(lambda x: 0 if x in ["legbyes", "byes"] else 1) * seasondf["total_run"]
seasondf["IsLegalBall"] = seasondf.extra_type.apply(lambda x: 0 if x in ["wides", "noballs"] else 1)
pcapdf = seasondf.groupby(["Season", "bowler"], as_index = False)[["IsBowlerWicket", "BowlerRun", "IsLegalBall"]].sum()
pcapdf["Economy"] = pcapdf["BowlerRun"] / pcapdf["IsLegalBall"] * 6
pcapdf.sort_values(["IsBowlerWicket", "Economy"], ascending = [False, True]).drop_duplicates("Season", keep = "first").sort_values("Season")

       Season         bowler  IsBowlerWicket  BowlerRun  IsLegalBall   Economy
84    2007/08  Sohail Tanvir              22        266          247  6.461538
174      2009       RP Singh              23        417          358  6.988827
284   2009/10        PP Ojha              21        429          353  7.291785
447      2011     SL Malinga              28        375          378  5.952381
537      2012       M Morkel              25        453          378  7.190476
629      2013       DJ Bravo              32        497          375  7.952000
779      2014      MM Sharma              23        452          323  8.396285
847      2015       DJ Bravo              26        426          314  8.140127
938      2016        B Kumar              23        490          396  7.424242
1048     2017        B Kumar              26        369          314  7.050955
1144     2018         AJ Tye              24        448          336  8.000000
1269     2019    Imran Tahir              26        

Match Result from- https://www.mykhel.com/cricket/ipl-purple-cap-winners-list-s4/

# Task Q-7 — Best Death-Overs Bowler

**Goal:** Find the best bowler in the death overs (overs 16–20), ranked by most wickets and tie-broken by least economy.

### `Q-7:` Best bowler in death overs. 
*Note: Have taken most no of wickets in case of tie with least economy*

Death Overs - [16-20]

## 2.2 Approach — reuse the engineered columns on a filtered slice

The beauty here is reuse: the `IsBowlerWicket`, `BowlerRun`, and `IsLegalBall` columns engineered in Q-6 already encode the correct counting rules, so the death-overs task needs only a filter and the same aggregation.
```python
death_overs = seasondf[seasondf.overs >= 15]
pcapdf = death_overs.groupby('bowler', as_index=False)[
    ['IsBowlerWicket','BowlerRun','IsLegalBall']].sum()
pcapdf['Economy'] = pcapdf['BowlerRun'] / pcapdf['IsLegalBall'] * 6
pcapdf.sort_values(['IsBowlerWicket','Economy'], ascending=[False, True]).head()
```
The filter `overs >= 15` selects deliveries from the 16th over onward, since the `overs` column is zero-based (over 0 is the first over, so overs 15–19 are overs 16–20). Grouping by bowler *across all seasons* (no `Season` key this time) totals each bowler's career death-overs figures, and the same sort implements wickets-then-economy. The leaders — DJ Bravo (115 wickets at 9.36), SL Malinga (108 at 7.86), B Kumar (90 at 8.96), JJ Bumrah (80) — are the specialist death bowlers cricket fans expect. Note the deliberate contrast with Q-6: same derived features, but grouped by bowler instead of season and filtered to a phase of the innings. Always sanity-check zero-denominators before dividing.

In [ ]:
# code here
death_overs = seasondf[seasondf.overs>=15]
pcapdf = death_overs.groupby("bowler", as_index = False)[["IsBowlerWicket", "BowlerRun", "IsLegalBall"]].sum()
pcapdf["Economy"] = pcapdf["BowlerRun"] / pcapdf["IsLegalBall"] * 6
pcapdf.sort_values(["IsBowlerWicket", "Economy"], ascending = [False, True]).head()

         bowler  IsBowlerWicket  BowlerRun  IsLegalBall   Economy
91     DJ Bravo             115       2161         1385  9.361733
331  SL Malinga             108       1464         1117  7.863921
53      B Kumar              90       1752         1173  8.961637
148   JJ Bumrah              80       1614         1146  8.450262
339   SP Narine              67       1164          949  7.359326

# Task Q-8 — Season-Wise Batting Record Function

**Goal:** Write `bat_record_season(batsman)` returning a DataFrame indexed by `Season` with columns `Innings`, `TotalRuns`, `Avg`, `HighestScore`, `StrikeRate` — handling wides, no-balls, and how dismissals are attributed correctly.

### `Q-8` Batsman record season wise

Make a function which takes a input `batsman_name` and it returns a dataframe.
Columns of the data frame are - `['Season','Innings', 'TotalRuns', 'Avg', 'HighestScore','StrikeRate']`.
* In result make `Season` column as index.

* Avg - total_runs/ no of time got out. - player_out column will help.
* StrikeRate -(total_runs/ balls faced) * 100- wides are not included in batsman ball faced counts. No balls are included. -> Extra_type column will help
* Batsman Can score runs on No Balls.
* Batsman can get out on No Ball or Wides. And even while being on non-striker. Keep these things in mind before masking.

## 2.3 Approach — balls faced, dismissals, and per-season aggregation

A batter's ball count comes from `IsBatsmanBall = 1 if extra_type != 'wides' else 0` — wides are not counted against the batter (no-balls *are*, and a batter can score off them). The function filters `batterdf[batterdf.batter == batsman]`, then computes `IsBatsmanOut = (bdf.batter == bdf.player_out)` so only the striker's own dismissals count (a batter cannot be out while at the non-striker's end).

The aggregation happens in two layers. First, group by `['Season','ID']` and sum `batsman_run`, `IsBatsmanBall`, and `IsBatsmanOut` — this collapses each innings to one row (Innings played = the count of these rows). Second, group by `Season` and aggregate with the dict form:
```python
df = df.groupby('Season').agg({'batsman_run':['sum','max'],
                               'IsBatsmanBall':'sum', 'IsBatsmanOut':'sum'})
df['Innings'] = innings
df['TotalRuns'] = df[('batsman_run','sum')]
df['Avg'] = df['TotalRuns'] / df[('IsBatsmanOut','sum')]
df['HighestScore'] = df[('batsman_run','max')]
df['StrikeRate'] = df['TotalRuns'] / df[('IsBatsmanBall','sum')] * 100
```
The multi-function aggregation creates **MultiIndex columns** (`('batsman_run','sum')`, etc.), accessed as tuples. The derived metrics are then Average (runs per dismissal), Highest Score (max in any innings), and Strike Rate (runs per 100 balls). `bat_record_season('MS Dhoni')` shows, for example, 2007/08: 14 innings, 414 runs, average 41.4, best 65, strike rate 133.5 — a compact career season-by-season profile. The caveats are exactly those the question warns about: no-balls can carry runs and even dismissals, non-striker run-outs must not be charged to the striker, and wides must be excluded from balls faced.

In [ ]:
# code here
batterdf = seasondf.copy()
batterdf["IsBatsmanBall"] = batterdf["extra_type"].apply(lambda x: 1 if x != "wides" else 0)

def bat_record_season(batsman):
    bdf = batterdf[batterdf.batter == batsman].copy()
    bdf["IsBatsmanOut"] = bdf.batter == bdf.player_out
    df = bdf.groupby(["Season", "ID"], as_index = False)[["batsman_run", "IsBatsmanBall", "IsBatsmanOut"]].sum()
    innings = df.groupby("Season").ID.count()
    df = df.groupby("Season").agg(
    {
        "batsman_run":["sum", "max"],
        "IsBatsmanBall": "sum",
        "IsBatsmanOut": "sum"
    })
    df["Innings"]= innings
    df["TotalRuns"] = df[("batsman_run", "sum")]
    df["Avg"] = df["TotalRuns"] / df[("IsBatsmanOut", "sum")]
    df["HighestScore"] = df[("batsman_run", "max")]
    df["StrikeRate"] = df["TotalRuns"] / df[("IsBatsmanBall", "sum")] * 100
    return df.drop(columns = ["batsman_run", "IsBatsmanBall", "IsBatsmanOut"])
bat_record_season("MS Dhoni")

        Innings TotalRuns         Avg HighestScore  StrikeRate
                                                              
Season                                                        
2007/08      14       414   41.400000           65  133.548387
2009         13       332   41.500000           58  127.203065
2009/10      11       287   31.888889           66  136.666667
2011         13       392   43.555556           70  158.704453
2012         17       358   32.545455           51  128.776978
2013         16       461   46.100000           67  162.897527
2014         15       371   74.200000           57  148.400000
2015         17       372   31.000000           53  121.967213
2016         12       284   40.571429           64  135.238095
2017         15       290   26.363636           61  116.000000
2018         15       455   75.833333           79  150.662252
2019         12       416  104.000000           84  134.627832
2020/21      12       200   25.000000           47  116

Verify at - https://www.iplt20.com/teams/chennai-super-kings/squad-details/1

# Task Q-9 — Player-of-the-Match Figures

**Goal:** Build a match-by-match DataFrame with columns `PlayerOfThematch`, `BattingFigure` (`runs/balls`), and `BowlingFigure` (`wickets/runs conceded`), one row per match.

### `Q-9` Using both dataset, make a dataframe as described below

Data Frame columns-> `['PlayerOfThematch', 'BattingFigure', 'BowlingFigure']`

* BattingFigure->`<runs>/<balls>`
* BowlingFigure->`<wicket>/<runs-conceded>`

DataFrame should have one record for each match.

Say 'V Kohli' got POM award then in dataset include his batting figure of that match. Say he scored 112runs in 76 balls. And he hasn't bowled so Bowling Figure will be NaN
```
PlayerOfThematch BattingFigure BowlingFigure
V Kohli          112/76         nan  

```


## 2.4 Batting and bowling figures, then an outer merge

First join the match-level winner onto the ball data: `df = balls.merge(matches[['ID','Player_of_Match']], on='ID')`. To get the POM player's **batting** figure, keep deliveries where `df.batter == df.Player_of_Match`, drop wides (balls-faced rule), group by `['ID','batter']` summing `batsman_run` and `IsBatsmanBall`, then format the pair as a string:
```python
batter['BattingFigure'] = batter[['batsman_run','IsBatsmanBall']].apply(
    lambda x: '/'.join(map(str, x.values)), axis=1)
```
The row-wise `apply(axis=1)` passes each row's two values, `map(str, ...)` stringifies them, and `'/'.join` glues them — producing `'158/73'` for BB McCullum's famous opening innings.

The **bowling** figure mirrors it: filter `df.bowler == df.Player_of_Match`, apply the same wicket/runs-rules as Q-6, group by `['ID','bowler']`, and format as `'2/11'`.

Finally, `batter.merge(bowler, on=['ID','PlayerOfMatch'], how='outer')` joins the two per-match tables. The **outer** join is the key choice: a POM who only batted appears in `batter` but not `bowler`, so an inner join would drop those matches entirely. With `outer`, the missing side becomes `NaN` — exactly the required output, e.g. V Kohli `112/76` with `BowlingFigure` NaN. Renaming the key to `PlayerOfMatch` in both halves is what allows the on-`['ID','PlayerOfMatch']` merge to line up.

In [ ]:
# code here
df = balls.merge(matches[["ID", "Player_of_Match"]], on="ID")
batterdf = df[df.batter == df.Player_of_Match].copy()
batterdf["IsBatsmanBall"] = batterdf["extra_type"].apply(lambda x: 1 if x != "wides" else 0)
batter = batterdf.groupby(["ID", "batter"], as_index=False)[["batsman_run", "IsBatsmanBall"]].sum()
batter["BattingFigure"] = batter[["batsman_run", "IsBatsmanBall"]].apply(lambda x: '/'.join(map(str, x.values)), axis = 1)
batter.rename(columns = {"batter": "PlayerOfMatch"}, inplace = True)
batter.head()

       ID PlayerOfMatch  batsman_run  IsBatsmanBall BattingFigure
0  335982   BB McCullum          158             73        158/73
1  335983    MEK Hussey          116             54        116/54
2  335985    MV Boucher           39             19         39/19
3  335986     DJ Hussey           38             43         38/43
4  335987     SR Watson           76             49         76/49

In [ ]:
bowlerdf = df[df.bowler == df.Player_of_Match].copy()
bowlerdf["IsBowlerWicket"] = bowlerdf["kind"].apply(lambda x: 1 if x in ["caught", 'caught and bowled', 'bowled', 'stumped','lbw', 'hit wicket'] else 0)
bowlerdf["BowlerRun"] = bowlerdf.extra_type.apply(lambda x: 0 if x in ["legbyes", "byes"] else 1) * bowlerdf["total_run"]
bowler = bowlerdf.groupby(["ID", "bowler"], as_index=False)[["IsBowlerWicket", "BowlerRun"]].sum()
bowler["BowlingFigure"] = bowler[["IsBowlerWicket", "BowlerRun"]].apply(lambda x: '/'.join(map(str, x.values)), axis = 1)
bowler.rename(columns = {"bowler": "PlayerOfMatch"}, inplace = True)
bowler.head()

       ID PlayerOfMatch  IsBowlerWicket  BowlerRun BowlingFigure
0  335984   MF Maharoof               2         11          2/11
1  335986     DJ Hussey               1         35          1/35
2  335987     SR Watson               0         39          0/39
3  335990     YK Pathan               2         20          2/20
4  335992     SR Watson               2         20          2/20

In [ ]:
batter.merge(bowler, on=["ID", "PlayerOfMatch"], how = "outer").drop(columns = ["batsman_run", "IsBatsmanBall", "IsBowlerWicket", "BowlerRun"]).head()

       ID PlayerOfMatch BattingFigure BowlingFigure
0  335982   BB McCullum        158/73           NaN
1  335983    MEK Hussey        116/54           NaN
2  335985    MV Boucher         39/19           NaN
3  335986     DJ Hussey         38/43          1/35
4  335987     SR Watson         76/49          0/39

# 3. Iris Dataset — Merging Many Fragments

The classic Iris flower dataset is provided in five pieces: sepal measurements, petal measurements, and three species tables. Reassembling them exercises multi-table joins and concatenation.

## **Questions Based on Iris Dataset**

- **Sepal All:** https://docs.google.com/spreadsheets/d/e/2PACX-1vT58ekmHTwptX7Bs4QOy6YByA1HMvYTACeeIjrKhHE0Pg1K_3egewHMKMh02zN9D5-yHVXfvuaa3s5u/pub?gid=2028782809&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating this sub-dataset.
    - **Id:** Id of the records.
    - **SepalLengthCm:** Sepal length of flowers in cm
    - **SepalWidthCm:** Sepal width of flowers in cm

- **Petal All:** https://docs.google.com/spreadsheets/d/e/2PACX-1vQinLXShrOz4ExNaW1bSQVuvbbhIzJW7G0kkkD2SvqSD6STjLrQQiftgI7BGe10sBZi0CNr2_sJpQAz/pub?gid=1580010789&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating this sub-dataset.
    - **Id:** Id of the records.
    - **PetalLengthCm:** Petal length of flowers in cm
    - **PetalWidthCm:** Petal width of flowers in cm

- **Iris Virginica:** https://docs.google.com/spreadsheets/d/e/2PACX-1vSK39MwduGPHYNgw5yViezoLYCVDKMCWIHzjnt3GZNaxHPFOQLr2q6no_tyqTsOk-VfXleslfGVe9eJ/pub?gid=314231613&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

- **Iris Versicolor:** https://docs.google.com/spreadsheets/d/e/2PACX-1vTcSFgLnabqIrgIc5WlwvnbbvyyJsgZjR-0E0-4TR-5aHgv_0EP6yNWglkkls3AXM2qHCR5VYzWCoTM/pub?gid=715607857&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

- **Iris Setosa:** https://docs.google.com/spreadsheets/d/e/2PACX-1vSjqJpdgy2X_oDUUqQ0sSaFKqnnf8MYU4KgJSYgHaHmq0Wb1weMOsJXh-rICHmkLcaTkOwzMYLeh959/pub?gid=2003684803&single=true&output=csv
    - **Unnamed 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

## 3.0 The five Iris fragments

Five CSVs are loaded from published URLs:
- `sepal_all` — `Id`, `SepalLengthCm`, `SepalWidthCm`
- `petal_all` — `Id`, `PetalLengthCm`, `PetalWidthCm`
- `virginica`, `versicolor`, `setosa` — each `Id` plus `Species`

Every file also carries an `Unnamed: 0` column — the artefact Pandas creates when a CSV is written with its index and then re-read without `index_col=0`. `Id` is the shared key joining all five. This is a deliberately 'normalised' presentation: measurements in one pair of tables, species labels in another — so reconstructing the familiar flat Iris frame requires joining by `Id` and concatenating the three species blocks. Recognising that `Unnamed: 0` is junk and that `Id` is the common key is half the battle before any code is written.

In [ ]:

sepal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT58ekmHTwptX7Bs4QOy6YByA1HMvYTACeeIjrKhHE0Pg1K_3egewHMKMh02zN9D5-yHVXfvuaa3s5u/pub?gid=2028782809&single=true&output=csv")
petal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQinLXShrOz4ExNaW1bSQVuvbbhIzJW7G0kkkD2SvqSD6STjLrQQiftgI7BGe10sBZi0CNr2_sJpQAz/pub?gid=1580010789&single=true&output=csv")

virginica = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSK39MwduGPHYNgw5yViezoLYCVDKMCWIHzjnt3GZNaxHPFOQLr2q6no_tyqTsOk-VfXleslfGVe9eJ/pub?gid=314231613&single=true&output=csv")
versicolor = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vTcSFgLnabqIrgIc5WlwvnbbvyyJsgZjR-0E0-4TR-5aHgv_0EP6yNWglkkls3AXM2qHCR5VYzWCoTM/pub?gid=715607857&single=true&output=csv")
setosa = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSjqJpdgy2X_oDUUqQ0sSaFKqnnf8MYU4KgJSYgHaHmq0Wb1weMOsJXh-rICHmkLcaTkOwzMYLeh959/pub?gid=2003684803&single=true&output=csv")


# Task Q-9 — Average Sepal vs Petal Length Bar Chart

**Goal:** Plot a bar chart comparing the average sepal length of Virginica with the average petal length of Setosa.

### `Q-9:` Plot a bar chart of the average Sepal Length  of Virginica and average Petal length of Setosa flower.

## 3.1 Approach — merge to get measurements, build a small frame, plot

Each species table holds only `Id` and `Species`, so to get measurements you join it with the matching measurement file: `pd.merge(virginica, sepal_all, on='Id')['SepalLengthCm'].mean()` attaches sepal measurements to the Virginica rows and averages the length; `pd.merge(setosa, petal_all, on='Id')['PetalLengthCm'].mean()` does the analogous petal calculation for Setosa. Both merges are keyed on `Id`, a one-to-one join.

The two scalars are then placed in a tiny purpose-built frame — `pd.DataFrame({'kind':['Sepal Length','Petal Length'], 'Length':[sepal, petal]})` — and plotted with `.plot(x='kind', kind='bar')`. Building a small summary frame and plotting it is the standard way to chart a handful of aggregates; the plotting accessor handles axis labels and the categorical x-axis automatically. The resulting bars contrast two different flowers on two different dimensions, and the comparison is meaningful because both values are in centimetres. The lesson: when your data is spread across several tables, merge first to attach the columns you need, *then* aggregate and visualise.

In [ ]:
# code here
sepal = pd.merge(virginica, sepal_all, on="Id")["SepalLengthCm"].mean()
petal = pd.merge(setosa, petal_all, on="Id")["PetalLengthCm"].mean()
pd.DataFrame({"kind": ["Sepal Length", "Petal Length"], "Length": [sepal, petal]}).plot(x="kind", kind = "bar")

<AxesSubplot:xlabel='kind'>

<Figure size 640x480 with 1 Axes>

# Task Q-10 — Assemble the Complete, Shuffled Dataset

**Goal:** Combine `virginica`, `versicolor`, `setosa`, `sepal_all`, and `petal_all` into one frame with columns `Id`, `Species`, `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`, then shuffle so `Id` is not ordered.

### `Q-10:` Create the complete dataset by uisng the below datasets:
- virginica
- versicolor
- setosa
- sepal all
- petal all

This dataset should have these below column names in order:
1. Id
2. Species
3. SepalLengthCm
4. SepalWidthCm
5. PetalLengthCm
6. PetalWidthCm

Also, the dataset should be shuffled means the `Id` column should not be in increasing or decreasing order. So, make a dataset which has the shuffled Id column. You can use `DataFrame.sample()` method to shuffle. 

## 3.2 Approach — chained merges, concat, and sample

For each species, two merges are chained: first join the species table to `sepal_all` on `Id` (bringing in sepal measurements), then join the result to `petal_all` on `Id` (bringing in petal measurements). At each step the redundant `Unnamed: 0` columns are dropped — `drop(columns=['Unnamed: 0_x','Unnamed: 0_y'])` after the first merge (the suffixes appear because both frames had a same-named junk column) and `drop(columns=['Unnamed: 0'])` after the second. Doing this per species produces three complete frames `v`, `s`, and `ver`.

`pd.concat([v, s, ver])` then stacks the three vertically into one 150-row frame with the desired column order (`Id`, `Species`, then the four measurements), and `.sample(150)` shuffles it. `sample(n)` draws `n` rows **without replacement** by default — here all 150, so the result is simply a random permutation, which satisfies the requirement that `Id` not be in ascending order. A fixed `random_state` would make the shuffle reproducible; `.sample(frac=1)` is the equivalent idiom for 'shuffle everything'.

Watch the merging pitfalls: the same `Id` may originate the same `Unnamed: 0` name in multiple frames (hence the `_x`/`_y` suffixes), and if the three species tables did not cover disjoint `Id` ranges you could see duplicates. For the Iris data the split is clean, so the concatenation yields the canonical 150 rows, each tagged with its species.

In [ ]:
# code here
v_sepal = pd.merge(virginica, sepal_all, on="Id").drop(columns= ["Unnamed: 0_x", "Unnamed: 0_y"])
v = pd.merge(v_sepal, petal_all, on="Id").drop(columns= ["Unnamed: 0"])

s_sepal = pd.merge(setosa, sepal_all, on="Id").drop(columns= ["Unnamed: 0_x", "Unnamed: 0_y"])
s = pd.merge(s_sepal, petal_all, on="Id").drop(columns= ["Unnamed: 0"])

ver_sepal = pd.merge(versicolor, sepal_all, on="Id").drop(columns= ["Unnamed: 0_x", "Unnamed: 0_y"])
ver = pd.merge(ver_sepal, petal_all, on="Id").drop(columns= ["Unnamed: 0"])

pd.concat([v, s, ver]).sample(150)

     Id          Species  SepalLengthCm  SepalWidthCm  PetalLengthCm  \
20  121   Iris-virginica            6.9           3.2            5.7   
7   108   Iris-virginica            7.3           2.9            6.3   
31  132   Iris-virginica            7.9           3.8            6.4   
11   62  Iris-versicolor            5.9           3.0            4.2   
25   26      Iris-setosa            5.0           3.0            1.6   
..  ...              ...            ...           ...            ...   
27   28      Iris-setosa            5.2           3.5            1.5   
39   90  Iris-versicolor            5.5           2.5            4.0   
46   97  Iris-versicolor            5.7           2.9            4.2   
31   32      Iris-setosa            5.4           3.4            1.5   
19   70  Iris-versicolor            5.6           2.5            3.9   

    PetalWidthCm  
20           2.3  
7            1.8  
31           2.0  
11           1.5  
25           0.2  
..           ...  
27

# Task Q-11 — Sepal and Petal Width Extremes per Species

**Goal:** Report the minimum and maximum sepal width and petal width for Setosa and Versicolor, using a merged dataset and `groupby('Species')`.

### `Q-11:` Find out the maximum and minimum sepal width and petal width of Setosa and Versicolor. To do this:
- First create a dataset with merging the required datasets
- After that, use `groupby` to create groups based on the "Species" column.
- Then find out which are asked in this question.


The output should be like this:
```bash
Minimum Sepal width of Setosa is 2.3
Maximum Sepal width of Setosa is 4.4

**************************************************

Minimum Sepal width of Versicolor is 2.0
Maximum Sepal width of Versicolor is 3.4

**************************************************
```

## 3.3 Approach — group and aggregate, or slice the prepared frames

Because the previous question already assembled complete per-species frames (`s` for Setosa, `ver` for Versicolor), the direct solution prints the four extremes from each: `s.SepalWidthCm.min()`, `s.SepalWidthCm.max()`, `s.PetalWidthCm.min()`, `s.PetalWidthCm.max()`, and the same four for `ver`. The output gives Sepal width 2.3–4.4 and Petal width 0.1–0.6 for Setosa, and Sepal 2.0–3.4, Petal 1.0–1.6 for Versicolor — figures that align with the well-known Iris statistics (and confirm Setosa's tiny petals make it linearly separable from the other two species).

The question explicitly asks you to use `groupby`, which points at the more scalable formulation: build one merged frame containing a `Species` column, then `merged.groupby('Species')[['SepalWidthCm','PetalWidthCm']].agg(['min','max'])`. That produces a tidy table of extremes for *all* species in a single call rather than four hard-coded lines per species — far better when new species or columns appear. The comparison is also the classic preview of classification: Setosa's petal width rarely exceeds 0.6 cm while Versicolor's never drops below 1.0, a clean gap a simple threshold could exploit. This final task closes the notebook by combining merging, grouping, and summary statistics into one interpretable result.

In [ ]:
# code here
print(f"Minimum Sepal width of Setosa is {s.SepalWidthCm.min()}")
print(f"Maximum Sepal width of Setosa is {s.SepalWidthCm.max()}")
print(f"Minimum Petal width of Setosa is {s.PetalWidthCm.min()}")
print(f"Maximum Petal width of Setosa is {s.PetalWidthCm.max()}")
print(f"Minimum Sepal width of Versicolor is {ver.SepalWidthCm.min()}")
print(f"Maximum Sepal width of Versicolor is {ver.SepalWidthCm.max()}")
print(f"Minimum Petal width of Versicolor is {ver.PetalWidthCm.min()}")
print(f"Maximum Petal width of Versicolor is {ver.PetalWidthCm.max()}")

Minimum Sepal width of Setosa is 2.3
Maximum Sepal width of Setosa is 4.4
Minimum Petal width of Setosa is 0.1
Maximum Petal width of Setosa is 0.6
Minimum Sepal width of Versicolor is 2.0
Maximum Sepal width of Versicolor is 3.4
Minimum Petal width of Versicolor is 1.0
Maximum Petal width of Versicolor is 1.8


## Summary

Task 20 combined concatenation, merging, grouping, and plotting across three datasets. On the quarterly sales data you stacked three files with `pd.concat(..., keys=['Q1','Q2','Q3'])`, reset the resulting MultiIndex, merged the item-name lookup, converted `$` prices with an `apply`, and answered three business questions — most-sold item, top-revenue item, and average order price per quarter — each via group-aggregate-sort-dedupe. On the IPL data you merged season info onto ball-by-ball records, engineered `IsBowlerWicket`, `BowlerRun`, and `IsLegalBall` to encode cricket rules, and derived the Purple Cap winner per season (wickets desc, economy asc), the best career death-overs bowler, a reusable `bat_record_season(batsman)` function (innings, runs, average, best, strike rate with correct wide/no-ball handling), and a per-match Player-of-the-Match table combining batting (`runs/balls`) and bowling (`wickets/runs`) figures via an **outer** merge. On Iris you reassembled five fragments by merging on `Id`, concatenated the species blocks, shuffled with `sample()`, and used `groupby('Species')` to compare sepal/petal width extremes. Master motifs: keys give provenance, merges attach context, derived columns encode domain rules, and grouping answers the question.